In [ ]:
import time
import json
import numpy as np
import onnxruntime as ort
from pathlib import Path
from transformers import AutoTokenizer

# ==========================================
# 1. SETUP PATH LOKAL WINDOWS
# ==========================================
MODEL_DIR = Path(r"D:\Data Ade\Antigravity 2.0\Hackaton\Human_Firewall\results nlp")
ONNX_FILE = MODEL_DIR / "model.onnx"
CONFIG_FILE = MODEL_DIR / "threshold_config.json"

# Load Threshold Config
if CONFIG_FILE.exists():
    with open(CONFIG_FILE, "r") as f:
        config = json.load(f)
    THR_BLOCK = config.get("thr_auto_block", 0.9920)
    LABEL_ORDER = config.get("label_order", ["benign", "phishing", "malware", "other"])
else:
    THR_BLOCK = 0.9920
    LABEL_ORDER = ["benign", "phishing", "malware", "other"]

# ==========================================
# 2. INISIALISASI TOKENIZER & ONNX SESSION (INTEL ARC / CPU)
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# Provider untuk Intel Core Ultra 5 / Intel Arc Graphics:
# Gunakan 'DmlExecutionProvider' jika menginstal onnxruntime-directml
# Atau 'CPUExecutionProvider' untuk running di CPU Ultra 5
providers = [
    'DmlExecutionProvider',
    'CPUExecutionProvider'
]

session = ort.InferenceSession(str(ONNX_FILE), providers=providers)
print(f"ONNX Session Active Provider: {session.get_providers()[0]}")

# ==========================================
# 3. HELPER & INFERENCE FUNCTION
# ==========================================
def head_tail_truncate(url_str, max_len=200):
    tokens = tokenizer.tokenize(str(url_str))
    if len(tokens) <= max_len - 2:
        return str(url_str)
    head_tokens = tokens[:75]
    tail_tokens = tokens[-(max_len - 2 - 75):]
    return tokenizer.convert_tokens_to_string(head_tokens + tail_tokens)

def predict_url(url_string):
    t0 = time.perf_counter()
    
    truncated_url = head_tail_truncate(url_string)
    inputs = tokenizer(
        truncated_url,
        truncation=True,
        padding="max_length",
        max_length=200,
        return_tensors="np"
    )
    
    onnx_inputs = {
        "input_ids": inputs["input_ids"].astype(np.int64),
        "attention_mask": inputs["attention_mask"].astype(np.int64)
    }
    
    logits = session.run(None, onnx_inputs)[0]
    
    # Softmax
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probas = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
    
    pred_idx = int(np.argmax(probas, axis=1)[0])
    p_malicious = float(1.0 - probas[0][0])
    latency_ms = (time.perf_counter() - t0) * 1000
    
    action = "BLOCK" if p_malicious >= THR_BLOCK else "ALLOW"
    
    return {
        "url": url_string,
        "predicted_class": LABEL_ORDER[pred_idx],
        "p_malicious": round(p_malicious, 4),
        "action": action,
        "latency_ms": round(latency_ms, 3)
    }

# ==========================================
# 4. TESTING RUN
# ==========================================
test_urls = [
    "https://www.google.com/search?q=python+tutorial",
    "http://login.paypal-security-update.account-verify.xyz/login.php",
    "http://slot-gacor-maxwin-olympus.site/register",
    "https://mandiri-utama-finance.co.id/layanan"
]

print(f"\nLoaded Threshold Auto-Block: P(malicious) >= {THR_BLOCK:.4f}\n")
for u in test_urls:
    res = predict_url(u)
    print(f"URL     : {res['url']}")
    print(f"Pred    : {res['predicted_class']:8s} | P(malicious): {res['p_malicious']:.4f} | Action: {res['action']:5s} | Latency: {res['latency_ms']} ms\n")

: 